<a href="https://colab.research.google.com/github/worldterminator/worldterminator/blob/main/diss%20data%20ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#NOLA 311 OPCD -Present & 12-18

In [ ]:
import requests
import pandas as pd

In [ ]:
API_URL = "https://data.nola.gov/resource/2jgv-pqrq.json"

KEEP_COLS = [
    "service_request",
    "request_type",
    "date_created",
    "case_close_date",
    "request_status",
    "responsible_agency",
    "address_councildis",
    "rowid",
    "final_x",
    "final_y",
    "longitude",
    "latitude",
    "geocoded_column",
]

test endpoint and the wanted fields

In [ ]:
params = {

    "$select": ",".join(KEEP_COLS),

    "$limit": 10

}
r = requests.get(API_URL, params=params, timeout=120)

r.raise_for_status()

df_test = pd.DataFrame(r.json())
print(df_test.shape)
df_test.head()

(10, 13)


,service_request,request_type,date_created,case_close_date,request_status,responsible_agency,rowid,longitude,latitude,geocoded_column,address_councildis,final_x,final_y
0,2021-847416,Mayor's Request,2021-12-10T21:22:27.000,2022-07-08T09:30:48.000,Closed,Executive Office of the Mayor,847416,0.0,0.0,"{'latitude': '0.0', 'longitude': '0.0'}",NaN,NaN,NaN
1,2022-858955,Tax and Revenue,2022-02-04T15:39:45.000,2022-02-22T01:09:55.000,Closed,Bureau of Revenue,858955,0.0,0.0,"{'latitude': '0.0', 'longitude': '0.0'}",NaN,NaN,NaN
2,2024-1145120,Traffic Safety,2024-10-30T12:18:29.000,NaN,Pending,Department of Public Works,1145120,-90.12458247242685,29.978768001649772,"{'latitude': '29.978768001649772', 'longitude'...",A,3663515.88941,539789.54625
3,2024-1145799,Trash/Recycling,2024-11-02T07:58:56.000,2024-11-02T04:09:40.000,Closed,Department of Sanitation,1145799,-90.091290391251,29.940758205316882,"{'latitude': '29.940758205316882', 'longitude'...",B,3674205.35803,526080.457854
4,2024-1145838,Roads and Streets,2024-11-02T15:10:10.000,NaN,Pending,Department of Public Works,1145838,-90.10896265604138,29.988242136809447,"{'latitude': '29.988242136809447', 'longitude'...",A,3668424.02055,543287.21947


In [ ]:
df_test.columns.tolist()

['service_request',
 'request_type',
 'date_created',
 'case_close_date',
 'request_status',
 'responsible_agency',
 'rowid',
 'longitude',
 'latitude',
 'geocoded_column',
 'address_councildis',
 'final_x',
 'final_y']

then full paginated request, for repeating

In [ ]:
LIMIT = 50000
offset = 0
parts = []

while True:
    params = {
        "$select": ",".join(KEEP_COLS),
        "$where": "date_created >= '2012-01-01T00:00:00'", #from 2012 onward
        "$limit": LIMIT,
        "$offset": offset,
        "$order": "rowid"
    }

    r = requests.get(API_URL, params=params, timeout=120)
    r.raise_for_status()

    rows = r.json()

    if not rows:
        break

    batch = pd.DataFrame(rows)
    parts.append(batch)

    offset += len(batch)
    print(f"Downloaded {offset:,} rows")

df_311 = pd.concat(parts, ignore_index=True)

print(f"\nDownload complete!")
print(f"Rows: {len(df_311):,}")
print(f"Columns: {len(df_311.columns)}")

Downloaded 50,000 rows
Downloaded 100,000 rows
Downloaded 150,000 rows
Downloaded 200,000 rows
Downloaded 250,000 rows
Downloaded 300,000 rows
Downloaded 350,000 rows
Downloaded 400,000 rows
Downloaded 450,000 rows
Downloaded 500,000 rows
Downloaded 550,000 rows
Downloaded 600,000 rows
Downloaded 650,000 rows
Downloaded 700,000 rows
Downloaded 750,000 rows
Downloaded 800,000 rows
Downloaded 850,000 rows
Downloaded 900,000 rows
Downloaded 950,000 rows
Downloaded 1,000,000 rows
Downloaded 1,020,471 rows

Download complete!
Rows: 1,020,471
Columns: 13


In [ ]:
print("Shape:", df_311.shape)

print("\nDate range:")
print(df_311["date_created"].min(), "to", df_311["date_created"].max())

print("\nDuplicate IDs:")
print("rowid:", df_311["rowid"].duplicated().sum())
print("service_request:", df_311["service_request"].duplicated().sum())

print("\nMissingness:")
print(df_311.isna().sum().sort_values(ascending=False))

print("\nTop request types:")
print(df_311["request_type"].value_counts(dropna=False).head(20))

lon = pd.to_numeric(df_311["longitude"], errors="coerce")
lat = pd.to_numeric(df_311["latitude"], errors="coerce")

valid_coord = (
    lon.notna() &
    lat.notna() &
    (lon != 0) &
    (lat != 0)
)

print("\nCoordinate check:")
print("Valid coordinates:", valid_coord.sum())
print("Valid coordinate %:", f"{valid_coord.mean():.2%}")
print("0,0 coordinates:", ((lon == 0) & (lat == 0)).sum())

Shape: (1020471, 13)

Date range:
2019-01-01T21:33:04.000 to 2026-08-17T23:24:34.000

Duplicate IDs:
rowid: 0
service_request: 0

Missingness:
address_councildis    329015
case_close_date       232542
final_y                40891
final_x                40891
responsible_agency      6635
request_type            1757
request_status             0
service_request            0
date_created               0
rowid                      0
longitude                  0
latitude                   0
geocoded_column            0
dtype: int64

Top request types:
request_type
Trash/Recycling                                475160
Property Maintenance                            99096
Abandoned Vehicles                              91900
Roads/Drainage                                  63309
Traffic Signals/Signs/Striping/Streetlights     58269
Streetlights                                    36963
Parks & Parkways                                31019
Roads and Streets                               29541
Dr

In [ ]:
df_311.to_parquet("311_OPCD_2019plus_raw_selected.parquet", index=False)

In [ ]:
from google.colab import files
files.download("311_OPCD_2019plus_raw_selected.parquet")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
311 OPCD 2012-2018

In [ ]:
API_URL_ARCHIVE = "https://data.nola.gov/resource/3iz8-nghx.json"

KEEP_COLS_ARCHIVE = [
    "ticket_id",
    "issue_type",
    "ticket_created_date_time",
    "ticket_closed_date_time",
    "ticket_status",
    "council_district",
    "longitude",
    "latitude",
    "location",
    "geom",
]

In [ ]:
params = {
    "$select": ",".join(KEEP_COLS_ARCHIVE),
    "$limit": 10
}

r = requests.get(API_URL_ARCHIVE, params=params, timeout=120)
r.raise_for_status()

df_311_archive_test = pd.DataFrame(r.json())

print(df_311_archive_test.shape)
df_311_archive_test.head()

(10, 9)


,ticket_id,issue_type,ticket_created_date_time,ticket_closed_date_time,ticket_status,council_district,longitude,latitude,location
0,101000847991,Street Light,2018-08-21T14:03:55.000,2018-08-28T09:38:35.000,Closed,A,-90.1157700850457,29.9202224971517,"{'latitude': '29.9202224971517', 'longitude': ..."
1,101000847903,Large Item Trash/Garbage Pickup,2018-08-21T12:29:40.000,2018-08-28T10:51:27.000,Closed,B,-90.1019344489206,29.950804540038,"{'latitude': '29.950804540038', 'longitude': '..."
2,101000849742,Pothole/Roadway Surface Repair,2018-08-24T14:41:00.000,2018-08-28T09:38:50.000,Closed,E,-90.0028355253192,30.0271568286374,"{'latitude': '30.0271568286374', 'longitude': ..."
3,101000847653,Trash/Garbage Pickup,2018-08-20T16:28:36.000,2018-08-28T14:03:28.000,Closed,A,-90.1045471973296,29.9699067477013,"{'latitude': '29.9699067477013', 'longitude': ..."
4,101000850848,Code Enforcement General Request,2018-08-28T11:34:00.000,2018-08-28T13:02:24.000,Closed,C,-90.0491191933234,29.9446163523183,"{'latitude': '29.9446163523183', 'longitude': ..."


In [ ]:
LIMIT = 50000
offset = 0
parts = []

while True:
    params = {
        "$select": ",".join(KEEP_COLS_ARCHIVE),
        "$limit": LIMIT,
        "$offset": offset,
        "$order": "ticket_created_date_time,ticket_id"
    }

    r = requests.get(API_URL_ARCHIVE, params=params, timeout=120)
    r.raise_for_status()

    rows = r.json()

    if not rows:
        break

    batch = pd.DataFrame(rows)
    parts.append(batch)

    offset += len(batch)
    print(f"Downloaded {offset:,} rows")

df_311_archive = pd.concat(parts, ignore_index=True)

print("\nDownload complete!")
print(f"Rows: {len(df_311_archive):,}")
print(f"Columns: {len(df_311_archive.columns)}")

Downloaded 50,000 rows
Downloaded 100,000 rows
Downloaded 150,000 rows
Downloaded 200,000 rows
Downloaded 250,000 rows
Downloaded 300,000 rows
Downloaded 303,735 rows

Download complete!
Rows: 303,735
Columns: 9


In [ ]:
print("Shape:", df_311_archive.shape)

print("\nDate range:")
print(
    df_311_archive["ticket_created_date_time"].min(),
    "to",
    df_311_archive["ticket_created_date_time"].max()
)

print("\nDuplicate ticket_id:")
print(df_311_archive["ticket_id"].duplicated().sum())

print("\nMissingness:")
print(df_311_archive.isna().sum().sort_values(ascending=False))

print("\nTop issue types:")
print(
    df_311_archive["issue_type"]
    .value_counts(dropna=False)
    .head(20)
)

lon = pd.to_numeric(df_311_archive["longitude"], errors="coerce")
lat = pd.to_numeric(df_311_archive["latitude"], errors="coerce")

valid_coord = (
    lon.notna() &
    lat.notna() &
    (lon != 0) &
    (lat != 0)
)

print("\nCoordinate check:")
print("Valid coordinates:", valid_coord.sum())
print("Valid coordinate %:", f"{valid_coord.mean():.2%}")
print("0,0 coordinates:", ((lon == 0) & (lat == 0)).sum())

Shape: (303735, 9)

Date range:
2012-03-12T09:46:20.000 to 2019-01-02T14:28:00.000

Duplicate ticket_id:
1321

Missingness:
ticket_closed_date_time     25956
council_district            12807
ticket_id                       0
ticket_created_date_time        0
issue_type                      0
ticket_status                   0
longitude                       0
latitude                        0
location                        0
dtype: int64

Top issue types:
issue_type
Code Enforcement General Request       45126
Street Light                           36732
Trash/Garbage Pickup                   35680
Abandoned Vehicle Reporting/Removal    31798
Residential Recycling Programs         27501
Large Item Trash/Garbage Pickup        25070
Pothole/Roadway Surface Repair         16924
General Service Request                13195
Illegal Dumping Reporting              11764
Catch Basin Maintenance                 9480
Street Flooding/Drainage                9169
Tree Service                     

In [ ]:
dup = df_311_archive[ #given ticked_id duplicates
    df_311_archive["ticket_id"].duplicated(keep=False)
].sort_values("ticket_id")

dup.head(20)

,ticket_id,issue_type,ticket_created_date_time,ticket_closed_date_time,ticket_status,council_district,longitude,latitude,location
78,101000001000,Pothole/Roadway Surface Repair,2012-03-19T10:05:50.000,2012-11-08T16:24:34.000,Closed,B,-90.102347586828,29.9180961547502,"{'latitude': '29.9180961547502', 'longitude': ..."
79,101000001000,Pothole/Roadway Surface Repair,2012-03-19T10:05:50.000,2012-11-08T16:24:34.000,Closed,B,-90.102347586828,29.9180961547502,"{'latitude': '29.9180961547502', 'longitude': ..."
153,101000001441,Street Light,2012-03-21T09:52:21.000,2012-11-08T17:04:05.000,Closed,D,-90.0377474972926,29.9707357610986,"{'latitude': '29.9707357610986', 'longitude': ..."
154,101000001441,Street Light,2012-03-21T09:52:21.000,2012-11-08T17:04:05.000,Closed,D,-90.0377474972926,29.9707357610986,"{'latitude': '29.9707357610986', 'longitude': ..."
367,101000002297,Street Flooding/Drainage,2012-03-23T16:41:37.000,2012-04-24T08:59:21.000,Closed,A,-90.1041193534018,29.9794484455347,"{'latitude': '29.9794484455347', 'longitude': ..."
368,101000002297,Street Flooding/Drainage,2012-03-23T16:41:37.000,2012-04-24T08:59:21.000,Closed,A,-90.1041193534018,29.9794484455347,"{'latitude': '29.9794484455347', 'longitude': ..."
804,101000003641,Street Light,2012-03-30T12:09:09.000,2012-11-08T17:04:17.000,Closed,D,-90.0375946773703,29.9704552414518,"{'latitude': '29.9704552414518', 'longitude': ..."
805,101000003641,Street Light,2012-03-30T12:09:09.000,2012-11-08T17:04:17.000,Closed,D,-90.0375946773703,29.9704552414518,"{'latitude': '29.9704552414518', 'longitude': ..."
1224,101000004828,Street Light,2012-04-05T09:30:21.000,2013-10-16T17:01:25.000,Closed,A,-90.1320451686034,29.9433499480254,"{'latitude': '29.9433499480254', 'longitude': ..."
1223,101000004828,Street Light,2012-04-05T09:30:21.000,2013-10-16T17:01:25.000,Closed,A,-90.1320451686034,29.9433499480254,"{'latitude': '29.9433499480254', 'longitude': ..."


In [ ]:
cols = [
    "ticket_id",
    "issue_type",
    "ticket_created_date_time",
    "ticket_closed_date_time",
    "ticket_status",
    "council_district",
    "longitude",
    "latitude"
]

dup = df_311_archive[df_311_archive["ticket_id"].duplicated(keep=False)]
dup.duplicated(subset=cols).sum()

np.int64(1230)

ok,so 91 are not exact duplicates—at least differ from the other record with same id in one checked field

In [ ]:
dup.groupby("ticket_id")["issue_type"].nunique().value_counts()

,count
issue_type,
1,1274


In [ ]:
dup.groupby("ticket_id")[["longitude","latitude"]].agg(["min","max"])

longitude                             latitude  \
                            min                max               min   
ticket_id                                                              
101000001000   -90.102347586828   -90.102347586828  29.9180961547502   
101000001441  -90.0377474972926  -90.0377474972926  29.9707357610986   
101000002297  -90.1041193534018  -90.1041193534018  29.9794484455347   
101000003641  -90.0375946773703  -90.0375946773703  29.9704552414518   
101000004828  -90.1320451686034  -90.1320451686034  29.9433499480254   
...                         ...                ...               ...   
101000894048  -90.1091588802217  -90.1091588802217  29.9561969732936   
101000894051  -90.1091588802217  -90.1091588802217  29.9561969732936   
101000894052  -90.1091588802217  -90.1091588802217  29.9561969732936   
101000894441  -90.0556997667613  -90.0556997667613  29.9969945120915   
101000894444  -90.0556997667613  -90.0556997667613  29.9969945120915   

                                
                           max  
ticket_id                       
101000001000  29.9180961547502  
101000001441  29.9707357610986  
101000002297  29.9794484455347  
101000003641  29.9704552414518  
101000004828  29.9433499480254  
...                        ...  
101000894048  29.9561969732936  
101000894051  29.9561969732936  
101000894052  29.9561969732936  
101000894441  29.9969945120915  
101000894444  29.9969945120915  

[1274 rows x 4 columns]

In [ ]:
dup_num = dup.copy()

dup_num["longitude"] = pd.to_numeric(dup_num["longitude"], errors="coerce") #temporarily convert from strings to num for diagnostics
dup_num["latitude"] = pd.to_numeric(dup_num["latitude"], errors="coerce")

loc_check = (
    dup_num.groupby("ticket_id")
    .agg(
        lon_min=("longitude", "min"),
        lon_max=("longitude", "max"),
        lat_min=("latitude", "min"),
        lat_max=("latitude", "max"),
    )
)

loc_check["lon_diff"] = loc_check["lon_max"] - loc_check["lon_min"]
loc_check["lat_diff"] = loc_check["lat_max"] - loc_check["lat_min"]

loc_check[["lon_diff", "lat_diff"]].describe()

,lon_diff,lat_diff
count,1274.000000,1.274000e+03
mean,0.000001,9.250614e-07
std,0.000011,8.712533e-06
min,0.000000,0.000000e+00
25%,0.000000,0.000000e+00
50%,0.000000,0.000000e+00
75%,0.000000,0.000000e+00
max,0.000137,9.867382e-05


In [ ]:
loc_check.sort_values(
    "lon_diff",
    ascending=False
).head(20)

,lon_min,lon_max,lat_min,lat_max,lon_diff,lat_diff
ticket_id,,,,,,
101000574614,-90.036717,-90.036581,29.978911,29.978923,1.365920e-04,1.164096e-05
101000774719,-90.036717,-90.036581,29.978911,29.978923,1.365920e-04,1.164096e-05
101000324416,-90.036717,-90.036581,29.978911,29.978923,1.365920e-04,1.164096e-05
101000014627,-90.036717,-90.036581,29.978911,29.978923,1.365920e-04,1.164096e-05
101000358643,-90.036717,-90.036581,29.978911,29.978923,1.365920e-04,1.164096e-05
101000835872,-89.969436,-89.969304,30.056653,30.056752,1.317481e-04,9.832232e-05
101000639283,-89.969436,-89.969304,30.056653,30.056752,1.317481e-04,9.832232e-05
101000850675,-90.111214,-90.111166,29.954131,29.954219,4.731691e-05,8.782694e-05
101000841696,-90.111214,-90.111166,29.954131,29.954219,4.731691e-05,8.782694e-05


In [ ]:
loc_check.sort_values(
    "lat_diff",
    ascending=False
).head(20)

,lon_min,lon_max,lat_min,lat_max,lon_diff,lat_diff
ticket_id,,,,,,
101000357800,-90.120257,-90.120243,30.001040,30.001139,1.399249e-05,9.867382e-05
101000764422,-90.120257,-90.120243,30.001040,30.001139,1.399249e-05,9.867382e-05
101000835872,-89.969436,-89.969304,30.056653,30.056752,1.317481e-04,9.832232e-05
101000639283,-89.969436,-89.969304,30.056653,30.056752,1.317481e-04,9.832232e-05
101000882382,-90.111214,-90.111166,29.954131,29.954219,4.731691e-05,8.782694e-05
101000850675,-90.111214,-90.111166,29.954131,29.954219,4.731691e-05,8.782694e-05
101000841696,-90.111214,-90.111166,29.954131,29.954219,4.731691e-05,8.782694e-05
101000834485,-90.111214,-90.111166,29.954131,29.954219,4.731691e-05,8.782694e-05
101000689689,-90.111214,-90.111166,29.954131,29.954219,4.731691e-05,8.782694e-05


In [ ]:
worst = loc_check.sort_values(
    "lon_diff",
    ascending=False
).head(10).index

dup_num[
    dup_num["ticket_id"].isin(worst)
].sort_values("ticket_id")

,ticket_id,issue_type,ticket_created_date_time,ticket_closed_date_time,ticket_status,council_district,longitude,latitude,location
6543,101000014627,Residential Recycling Programs,2012-05-08T09:39:53.000,2012-05-31T09:07:53.000,Closed,D,-90.036717,29.978911,"{'latitude': '29.9789111901088', 'longitude': ..."
6544,101000014627,Residential Recycling Programs,2012-05-08T09:39:53.000,2012-05-31T09:07:53.000,Closed,D,-90.036581,29.978923,"{'latitude': '29.9789228310644', 'longitude': ..."
6545,101000014627,Residential Recycling Programs,2012-05-08T09:39:53.000,2012-05-31T09:07:53.000,Closed,D,-90.036581,29.978923,"{'latitude': '29.9789228310644', 'longitude': ..."
104408,101000324416,Rodent Complaint,2014-10-23T10:15:05.000,2014-10-28T10:50:20.000,Closed,D,-90.036717,29.978911,"{'latitude': '29.9789111901088', 'longitude': ..."
104409,101000324416,Rodent Complaint,2014-10-23T10:15:05.000,2014-10-28T10:50:20.000,Closed,D,-90.036581,29.978923,"{'latitude': '29.9789228310644', 'longitude': ..."
104410,101000324416,Rodent Complaint,2014-10-23T10:15:05.000,2014-10-28T10:50:20.000,Closed,D,-90.036581,29.978923,"{'latitude': '29.9789228310644', 'longitude': ..."
113634,101000358643,Street Light,2015-02-03T11:41:27.000,2015-04-15T15:20:31.000,Closed,D,-90.036717,29.978911,"{'latitude': '29.9789111901088', 'longitude': ..."
113635,101000358643,Street Light,2015-02-03T11:41:27.000,2015-04-15T15:20:31.000,Closed,D,-90.036581,29.978923,"{'latitude': '29.9789228310644', 'longitude': ..."
113636,101000358643,Street Light,2015-02-03T11:41:27.000,2015-04-15T15:20:31.000,Closed,D,-90.036581,29.978923,"{'latitude': '29.9789228310644', 'longitude': ..."
180686,101000574614,Catch Basin Maintenance,2016-08-03T12:07:00.000,2017-03-23T11:21:10.000,Closed,D,-90.036717,29.978911,"{'latitude': '29.9789111901088', 'longitude': ..."


most of the duplications have pairs of coordinates in close distance, likely in the same street segments or blocks. should be fine; since any error is internal to their internal workflow.
*i'm really not trying to recover the true history of these tickets, but to determine exclusion criteria for a cleaner sample. i cannot guess the request type that best represents a ticket so

with the unit being a service request with one substantive classification occurring at one (similar enough) location, any tickets that fail to satisfy could defensibly be excluded

In [ ]:
'''A duplicated ticket is retained only if it can be represented as one clean request type at one sufficiently similar location.
Otherwise, exclude the entire ticket ID.'''
#pull rows belonging to duplicated ticket IDs
dup = df_311_archive[
    df_311_archive["ticket_id"].duplicated(keep=False)
].copy()

#convert coordinates only in this diagnostic copy
dup["longitude_num"] = pd.to_numeric(dup["longitude"], errors="coerce")
dup["latitude_num"] = pd.to_numeric(dup["latitude"], errors="coerce")


In [ ]:
#summarize each duplicated ticket ID
dup_summary = (
    dup.groupby("ticket_id")
    .agg(
        n_rows=("ticket_id", "size"),
        n_issue_types=("issue_type", "nunique"),
        lon_min=("longitude_num", "min"),
        lon_max=("longitude_num", "max"),
        lat_min=("latitude_num", "min"),
        lat_max=("latitude_num", "max"),
    )
)

dup_summary["lon_diff"] = dup_summary["lon_max"] - dup_summary["lon_min"]
dup_summary["lat_diff"] = dup_summary["lat_max"] - dup_summary["lat_min"]

In [ ]:
# define what counts as a materially different location
LOCATION_TOL = 0.0002

In [ ]:
 #flag ambiguous duplicated tickets
dup_summary["type_ambiguous"] = (
    dup_summary["n_issue_types"] > 1
)
dup_summary["location_ambiguous"] = (
    (dup_summary["lon_diff"] > LOCATION_TOL) |
    (dup_summary["lat_diff"] > LOCATION_TOL)
)
dup_summary["exclude"] = (
    dup_summary["type_ambiguous"] |
    dup_summary["location_ambiguous"]
)

In [ ]:
print("Duplicated ticket IDs:", len(dup_summary))

print(
    "Different request types:",
    dup_summary["type_ambiguous"].sum()
)

print(
    "Materially different locations:",
    dup_summary["location_ambiguous"].sum()
)

print(
    "Excluded duplicated ticket IDs:",
    dup_summary["exclude"].sum()
)

Duplicated ticket IDs: 1274
Different request types: 0
Materially different locations: 0
Excluded duplicated ticket IDs: 0


can report very cleanly now:
* 1,321 was the number of extra duplicated rows flagged earlier.
* 1,274 is the number of unique ticket_ids that occur more than once.

among all 1,274 duplicated ticket IDs:
* 0 have more than one issue_type
* 0 exceed our location-difference tolerance
* therefore 0 are ambiguous under the rule we defined.

can simply deduplicate to one row per ticket_id during cleaning ;3

In [ ]:
df_311_archive_clean = (
    df_311_archive
    .drop_duplicates(subset="ticket_id", keep="first")
    .copy()
)

print("Original rows:", len(df_311_archive))
print("Clean rows:", len(df_311_archive_clean))
print(
    "Remaining duplicate ticket IDs:",
    df_311_archive_clean["ticket_id"].duplicated().sum()
)

Original rows: 303735
Clean rows: 302414
Remaining duplicate ticket IDs: 0


In [ ]:
df_311_archive.to_parquet(
    "311_historic_2012_2018_raw_selected.parquet",
    index=False
)

df_311_archive_clean.to_parquet(
    "311_historic_2012_2018_clean.parquet",
    index=False
)

In [ ]:
import os

for f in [
    "311_historic_2012_2018_raw_selected.parquet",
    "311_historic_2012_2018_clean.parquet"
]:
    print(f"{f}: {os.path.getsize(f)/1024/1024:.2f} MB")

311_historic_2012_2018_raw_selected.parquet: 18.18 MB
311_historic_2012_2018_clean.parquet: 18.15 MB


In [ ]:
from google.colab import files

files.download("311_historic_2012_2018_raw_selected.parquet")
files.download("311_historic_2012_2018_clean.parquet")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

for next time—is parquet this nice?

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving 311_historic_2012_2018_clean.parquet to 311_historic_2012_2018_clean.parquet


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving 311_OPCD_2019plus_raw_selected.parquet to 311_OPCD_2019plus_raw_selected.parquet


In [ ]:
import pandas as pd
df_311 = pd.read_parquet(
    "311_OPCD_2019plus_raw_selected.parquet"
)

In [ ]:
df_311.shape

(1020471, 13)

In [ ]:
df_311.head()

,service_request,request_type,date_created,case_close_date,request_status,responsible_agency,rowid,final_x,final_y,longitude,latitude,geocoded_column,address_councildis
0,101000091247,Trash/Recycling,2019-01-01T21:33:04.000,2013-01-13T18:00:00.000,Closed,Department of Sanitation,194420,3666381.059,522999.1162,-90.1160965018,29.932516387,"{'latitude': '29.932516387', 'longitude': '-90...",None
1,101000091249,Traffic Signals/Signs/Striping/Streetlights,2019-01-01T21:33:04.000,2013-02-21T18:00:00.000,Closed,Department of Public Works,194421,3666381.059,522999.1162,-90.1160965018,29.932516387,"{'latitude': '29.932516387', 'longitude': '-90...",None
2,101000091254,Trash/Recycling,2019-01-01T21:33:04.000,2013-01-06T18:00:00.000,Closed,Department of Sanitation,194422,3676109.517,518719.0788,-90.0855317174,29.9204599165,"{'latitude': '29.9204599165', 'longitude': '-9...",None
3,101000091260,Traffic Signals/Signs/Striping/Streetlights,2019-01-01T21:33:04.000,2013-10-30T19:00:00.000,Closed,Department of Public Works,194423,3681460.37,538282.1624,-90.0679596593,29.9740905155,"{'latitude': '29.9740905155', 'longitude': '-9...",None
4,101000091265,Trash/Recycling,2019-01-01T21:33:04.000,2013-02-26T18:00:00.000,Closed,Department of Sanitation,194424,3704549.425,553750.596,-89.9944689044,30.0159016139,"{'latitude': '30.0159016139', 'longitude': '-8...",None


#NOLA CFS 2023 year inspection

In [ ]:
import requests
import pandas as pd

In [ ]:
API_URL_CFS = "https://data.nola.gov/resource/pc5d-tvaw.json"

KEEP_COLS_CFS = [
    "nopd_item",
    "type_",
    "typetext",
    "initialtype",
    "initialtypetext",
    "priority",
    "initialpriority",
    "timecreate",
    "selfinitiated",
    "block_address",
    "zip",
    "policedistrict",
    "location",
]

In [ ]:
params = {
    "$select": ",".join(KEEP_COLS_CFS),
    "$limit": 10
}

r = requests.get(API_URL_CFS, params=params, timeout=120)
r.raise_for_status()

df_cfs_test = pd.DataFrame(r.json())

print(df_cfs_test.shape)
df_cfs_test.head()

(10, 13)


,nopd_item,type_,typetext,initialtype,initialtypetext,priority,initialpriority,timecreate,selfinitiated,block_address,zip,policedistrict,location
0,A0599223,SEXOFF,SEX OFFENSE: GENERAL/MISC,ASLT,SIMPLE ASSAULT,2,1,2023-01-06T16:55:53.930,N,002XX Henry Clay Av,70118,2,"{'type': 'Point', 'coordinates': [-90.12764112..."
1,B0272423,TRESP,TRESPASSING,TRESP,TRESPASSING,2,2,2023-02-03T17:30:36.337,N,028XX St Claude Av,70117,5,"{'type': 'Point', 'coordinates': [-90.04784966..."
2,A0653023,THEFT,THEFT,BURGV,BURGLARY FROM VEHICLE,2,0,2023-01-07T07:05:08.077,N,080XX Trapier Av,70127,7,"{'type': 'Point', 'coordinates': [-89.98874997..."
3,A0700123,MENTAL,EMOTIONALLY DISTURBED PERSON,MENTAL,EMOTIONALLY DISTURBED PERSON,1,2,2023-01-07T17:15:17.230,N,070XX Bundy Rd,None,0,"{'type': 'Point', 'coordinates': [0, 0]}"
4,B0287723,WELFARE,WELFARE CHECK,WELFARE,WELFARE CHECK,1,1,2023-02-03T20:21:39.033,N,112XX King Richard Dr,70128,7,"{'type': 'Point', 'coordinates': [-89.95505013..."


In [ ]:
LIMIT = 50000
offset = 0
parts = []

while True:
    params = {
        "$select": ",".join(KEEP_COLS_CFS),
        "$limit": LIMIT,
        "$offset": offset,
        "$order": "timecreate,nopd_item"
    }

    r = requests.get(API_URL_CFS, params=params, timeout=120)
    r.raise_for_status()

    rows = r.json()

    if not rows:
        break

    batch = pd.DataFrame(rows)
    parts.append(batch)

    offset += len(batch)
    print(f"Downloaded {offset:,} rows")

df_cfs = pd.concat(parts, ignore_index=True)

print("\nDownload complete!")
print(f"Rows: {len(df_cfs):,}")
print(f"Columns: {len(df_cfs.columns)}")

Downloaded 50,000 rows
Downloaded 100,000 rows
Downloaded 150,000 rows
Downloaded 200,000 rows
Downloaded 250,000 rows
Downloaded 300,000 rows
Downloaded 325,091 rows

Download complete!
Rows: 325,091
Columns: 13


In [ ]:
print("Shape:", df_cfs.shape)

print("\nDate range:")
print(
    df_cfs["timecreate"].min(),
    "to",
    df_cfs["timecreate"].max()
)

print("\nDuplicate NOPD_Item:")
print(df_cfs["nopd_item"].duplicated().sum())

print("\nMissingness:")
print(df_cfs.isna().sum().sort_values(ascending=False))

print("\nSelf-initiated:")
print(df_cfs["selfinitiated"].value_counts(dropna=False))

print("\nTop initial types:")
print(
    df_cfs["initialtypetext"]
    .value_counts(dropna=False)
    .head(20)
)

print("\nTop current/final types:")
print(
    df_cfs["typetext"]
    .value_counts(dropna=False)
    .head(20)
)

print("\nInitial priority:")
print(df_cfs["initialpriority"].value_counts(dropna=False))

print("\nCurrent/final priority:")
print(df_cfs["priority"].value_counts(dropna=False))

print("\nPolice districts:")
print(df_cfs["policedistrict"].value_counts(dropna=False).sort_index())

Shape: (325091, 13)

Date range:
2023-01-01T00:01:10.957 to 2023-12-31T23:58:29.070

Duplicate NOPD_Item:
0

Missingness:
nopd_item          0
type_              0
typetext           0
initialtype        0
initialtypetext    0
priority           0
initialpriority    0
timecreate         0
selfinitiated      0
block_address      0
zip                0
policedistrict     0
location           0
dtype: int64

Self-initiated:
selfinitiated
N    225389
Y     99702
Name: count, dtype: int64

Top initial types:
initialtypetext
MISCELLANEOUS COMPLAINT                 65821
ACCIDENT - PROPERTY DAMAGE              15872
AREA CHECK                              15223
DISORDERLY CONDUCT                      14011
INVESTIGATION: POLICE DEPARTMENT        11534
VEHICLE STOLEN                           9948
THEFT                                    9664
ALARM: COMMERCIAL - BURGLARY             9333
SUSPICIOUS PERSON                        9038
BURGLARY FROM VEHICLE                    8456
DOMESTIC DISPUT

In [ ]:
print("\nLocation missing:")
print(df_cfs["location"].isna().sum())

print("\nSample locations:")
print(df_cfs["location"].dropna().head())


Location missing:
0

Sample locations:
0    {'type': 'Point', 'coordinates': [-90.02946537...
1    {'type': 'Point', 'coordinates': [-90.06666469...
2    {'type': 'Point', 'coordinates': [-90.02173891...
3    {'type': 'Point', 'coordinates': [-90.06759171...
4    {'type': 'Point', 'coordinates': [-90.0208964,...
Name: location, dtype: object


In [ ]:
'''valid_location = (
    df_cfs["location"]
    .fillna("")
    .str.startswith("POINT (")
)

print("\nValid-looking POINT locations:")
print(valid_location.sum())
print(f"Valid-looking location %: {valid_location.mean():.2%}")'''
#so actually not imported as WKT strings but already GeoJSON-like dict


Valid-looking POINT locations:
0.0
Valid-looking location %: nan%


In [ ]:
loc_ok = df_cfs["location"].apply(
    lambda x: (
        isinstance(x, dict)
        and x.get("type") == "Point"
        and isinstance(x.get("coordinates"), list)
        and len(x["coordinates"]) == 2
    )
)

print("Valid Point locations:", loc_ok.sum())
print(f"Valid location %: {loc_ok.mean():.2%}")

Valid Point locations: 325091
Valid location %: 100.00%


In [ ]:
print("\nInitial vs current type agreement:")
same_type = (
    df_cfs["initialtype"].fillna("") ==
    df_cfs["type_"].fillna("")
)

print("Same type:", same_type.sum())
print(f"Same type %: {same_type.mean():.2%}")


Initial vs current type agreement:
Same type: 247470
Same type %: 76.12%


In [ ]:
df_cfs.to_parquet(
    "CFS_2023_raw_selected.parquet",
    index=False
)

In [ ]:
df_cfs_public = df_cfs[
    df_cfs["selfinitiated"] == "N"
].copy()

print(df_cfs_public.shape)

(225389, 13)


In [ ]:
df_cfs_public.to_parquet(
    "CFS_2023_public_initiated_clean.parquet",
    index=False
)

# NOLA CFS Reusable Download

In [ ]:
import requests
import pandas as pd

In [ ]:
def download_cfs_year(api_url, year, keep_cols, limit=50000):
    offset = 0
    parts = []

    while True:
        params = {
            "$select": ",".join(keep_cols),
            "$limit": limit,
            "$offset": offset,
            "$order": "timecreate,nopd_item"
        }

        r = requests.get(api_url, params=params, timeout=120)
        r.raise_for_status()

        rows = r.json()

        if not rows:
            break

        batch = pd.DataFrame(rows)
        parts.append(batch)

        offset += len(batch)
        print(f"{year}: downloaded {offset:,} rows")

        # Stop if this was the last partial page
        if len(batch) < limit: #this way it stops at the last run? without having to first create an empty call
            break

    df = pd.concat(parts, ignore_index=True)

    print(f"\n{year} complete")
    print("Shape:", df.shape)
    print(
        "Date range:",
        df["timecreate"].min(),
        "to",
        df["timecreate"].max()
    )

    return df

In [ ]:
KEEP_COLS_CFS = [
    "nopd_item",
    "type_",
    "typetext",
    "initialtype",
    "initialtypetext",
    "priority",
    "initialpriority",
    "timecreate",
    "selfinitiated",
    "block_address",
    "zip",
    "policedistrict",
    "location",
]

In [ ]:
def basic_cfs_checks(df):

    print("Shape:", df.shape)

    print("\nDate range:")
    print(
        df["timecreate"].min(),
        "to",
        df["timecreate"].max()
    )

    print("\nDuplicate NOPD_Item:")
    print(df["nopd_item"].duplicated().sum())

    print("\nMissingness:")
    print(df.isna().sum().sort_values(ascending=False))

    print("\nSelf-initiated:")
    print(df["selfinitiated"].value_counts(dropna=False))

    print("\nTop initial types:")
    print(
        df["initialtypetext"]
        .value_counts(dropna=False)
        .head(20)
    )

    print("\nTop current/final types:")
    print(
        df["typetext"]
        .value_counts(dropna=False)
        .head(20)
    )

    print("\nInitial priority:")
    print(
        df["initialpriority"]
        .value_counts(dropna=False)
    )

    print("\nCurrent/final priority:")
    print(
        df["priority"]
        .value_counts(dropna=False)
    )

    print("\nPolice districts:")
    print(
        df["policedistrict"]
        .value_counts(dropna=False)
        .sort_index()
    )

    loc_ok = df["location"].apply(
        lambda x: (
            isinstance(x, dict)
            and x.get("type") == "Point"
            and isinstance(x.get("coordinates"), list)
            and len(x["coordinates"]) == 2
        )
    )

    print("\nValid Point locations:")
    print(loc_ok.sum())
    print(f"Valid location %: {loc_ok.mean():.2%}")

    same_type = (
        df["initialtype"] == df["type_"]
    )

    print("\nInitial vs current type agreement:")
    print("Same type:", same_type.sum())
    print(f"Same type %: {same_type.mean():.2%}")

In [ ]:
API_URL_CFS_2025 = "https://data.nola.gov/resource/4xwx-sfte.json"

df_cfs_2025 = download_cfs_year(
    API_URL_CFS_2025,
    2025,
    KEEP_COLS_CFS
)
df_cfs_2025.to_parquet(
    "CFS_2025_raw_selected.parquet",
    index=False
)

2025: downloaded 50,000 rows
2025: downloaded 100,000 rows
2025: downloaded 150,000 rows
2025: downloaded 200,000 rows
2025: downloaded 250,000 rows
2025: downloaded 300,000 rows
2025: downloaded 329,770 rows

2025 complete
Shape: (329770, 13)
Date range: 2025-01-01T00:01:31.770000 to 2025-12-31T23:59:42.233000


In [ ]:
API_URL_CFS_2024 = "https://data.nola.gov/resource/2zcj-b6ts.json"

df_cfs_2024 = download_cfs_year(
    API_URL_CFS_2024,
    2024,
    KEEP_COLS_CFS
)
df_cfs_2024.to_parquet(
    "CFS_2024_raw_selected.parquet",
    index=False
)

2024: downloaded 50,000 rows
2024: downloaded 100,000 rows
2024: downloaded 150,000 rows
2024: downloaded 200,000 rows
2024: downloaded 250,000 rows
2024: downloaded 300,000 rows
2024: downloaded 327,696 rows

2024 complete
Shape: (327696, 13)
Date range: 2024-01-01T00:00:29.933 to 2024-12-31T23:57:28.433


In [ ]:
API_URL_CFS_2023 = "https://data.nola.gov/resource/pc5d-tvaw.json"

df_cfs_2023 = download_cfs_year(
    API_URL_CFS_2023,
    2023,
    KEEP_COLS_CFS
)
df_cfs_2023.to_parquet(
    "CFS_2023_raw_selected.parquet",
    index=False
)

2023: downloaded 50,000 rows
2023: downloaded 100,000 rows
2023: downloaded 150,000 rows
2023: downloaded 200,000 rows
2023: downloaded 250,000 rows
2023: downloaded 300,000 rows
2023: downloaded 325,091 rows

2023 complete
Shape: (325091, 13)
Date range: 2023-01-01T00:01:10.957 to 2023-12-31T23:58:29.070


In [ ]:
basic_cfs_checks(df_cfs_2023)

Shape: (325091, 13)

Date range:
2023-01-01T00:01:10.957 to 2023-12-31T23:58:29.070

Duplicate NOPD_Item:
0

Missingness:
nopd_item          0
type_              0
typetext           0
initialtype        0
initialtypetext    0
priority           0
initialpriority    0
timecreate         0
selfinitiated      0
block_address      0
zip                0
policedistrict     0
location           0
dtype: int64

Self-initiated:
selfinitiated
N    225389
Y     99702
Name: count, dtype: int64

Top initial types:
initialtypetext
MISCELLANEOUS COMPLAINT                 65821
ACCIDENT - PROPERTY DAMAGE              15872
AREA CHECK                              15223
DISORDERLY CONDUCT                      14011
INVESTIGATION: POLICE DEPARTMENT        11534
VEHICLE STOLEN                           9948
THEFT                                    9664
ALARM: COMMERCIAL - BURGLARY             9333
SUSPICIOUS PERSON                        9038
BURGLARY FROM VEHICLE                    8456
DOMESTIC DISPUT

In [ ]:
API_URL_CFS_2022 = "https://data.nola.gov/resource/nci8-thrr.json"

df_cfs_2022 = download_cfs_year(
    API_URL_CFS_2022,
    2022,
    KEEP_COLS_CFS
)
df_cfs_2022.to_parquet(
    "CFS_2022_raw_selected.parquet",
    index=False
)

In [ ]:
API_URL_CFS_2021 = "https://data.nola.gov/resource/3pha-hum9.json"

df_cfs_2021 = download_cfs_year(
    API_URL_CFS_2021,
    2021,
    KEEP_COLS_CFS
)
df_cfs_2021.to_parquet(
    "CFS_2021_raw_selected.parquet",
    index=False
)

2021: downloaded 50,000 rows
2021: downloaded 100,000 rows
2021: downloaded 150,000 rows
2021: downloaded 200,000 rows
2021: downloaded 250,000 rows
2021: downloaded 300,000 rows
2021: downloaded 350,000 rows
2021: downloaded 400,000 rows
2021: downloaded 428,315 rows

2021 complete
Shape: (428315, 13)
Date range: 2021-01-01T00:01:06.450 to 2021-12-31T23:58:27.133


In [ ]:
API_URL_CFS_2020 = "https://data.nola.gov/resource/hp7u-i9hf.json"

df_cfs_2020 = download_cfs_year(
    API_URL_CFS_2020,
    2020,
    KEEP_COLS_CFS
)
df_cfs_2020.to_parquet(
    "CFS_2020_raw_selected.parquet",
    index=False
)

2020: downloaded 50,000 rows
2020: downloaded 100,000 rows
2020: downloaded 150,000 rows
2020: downloaded 200,000 rows
2020: downloaded 250,000 rows
2020: downloaded 300,000 rows
2020: downloaded 350,000 rows
2020: downloaded 400,000 rows
2020: downloaded 432,892 rows

2020 complete
Shape: (432892, 13)
Date range: 2020-01-01T00:00:34.983 to 2020-12-31T23:58:13.170


In [ ]:
API_URL_CFS_2019 = "https://data.nola.gov/resource/qf6q-pp4b.json"

df_cfs_2019 = download_cfs_year(
    API_URL_CFS_2019,
    2019,
    KEEP_COLS_CFS
)
df_cfs_2019.to_parquet(
    "CFS_2019_raw_selected.parquet",
    index=False
)

2019: downloaded 50,000 rows
2019: downloaded 100,000 rows
2019: downloaded 150,000 rows
2019: downloaded 200,000 rows
2019: downloaded 250,000 rows
2019: downloaded 300,000 rows
2019: downloaded 350,000 rows
2019: downloaded 400,000 rows
2019: downloaded 450,000 rows
2019: downloaded 487,362 rows

2019 complete
Shape: (487362, 13)
Date range: 2019-01-01T00:00:02.000 to 2020-03-31T23:59:32.000


In [ ]:
API_URL_CFS_2018 = "https://data.nola.gov/resource/9san-ivhk.json"

df_cfs_2018 = download_cfs_year(
    API_URL_CFS_2018,
    2018,
    KEEP_COLS_CFS
)
df_cfs_2018.to_parquet(
    "CFS_2018_raw_selected.parquet",
    index=False
)

2018: downloaded 50,000 rows
2018: downloaded 100,000 rows
2018: downloaded 150,000 rows
2018: downloaded 200,000 rows
2018: downloaded 250,000 rows
2018: downloaded 300,000 rows
2018: downloaded 350,000 rows
2018: downloaded 400,000 rows
2018: downloaded 450,000 rows
2018: downloaded 460,874 rows

2018 complete
Shape: (460874, 13)
Date range: 2018-01-01T00:00:43.000 to 2018-12-31T23:59:07.000


In [ ]:
API_URL_CFS_2017 = "https://data.nola.gov/resource/bqmt-f3jk.json"

df_cfs_2017 = download_cfs_year(
    API_URL_CFS_2017,
    2017,
    KEEP_COLS_CFS
)
df_cfs_2017.to_parquet(
    "CFS_2017_raw_selected.parquet",
    index=False
)

2017: downloaded 50,000 rows
2017: downloaded 100,000 rows
2017: downloaded 150,000 rows
2017: downloaded 200,000 rows
2017: downloaded 250,000 rows
2017: downloaded 300,000 rows
2017: downloaded 350,000 rows
2017: downloaded 400,000 rows
2017: downloaded 444,111 rows

2017 complete
Shape: (444111, 13)
Date range: 2017-01-01T00:00:14.000 to 2017-12-31T23:59:53.000


In [ ]:
API_URL_CFS_2016 = "https://data.nola.gov/resource/wgrp-d3ma.json" #16

df_cfs_2016 = download_cfs_year(
    API_URL_CFS_2016,
    2016,
    KEEP_COLS_CFS
)
df_cfs_2016.to_parquet(
    "CFS_2016_raw_selected.parquet",
    index=False
)

2016: downloaded 50,000 rows
2016: downloaded 100,000 rows
2016: downloaded 150,000 rows
2016: downloaded 200,000 rows
2016: downloaded 250,000 rows
2016: downloaded 300,000 rows
2016: downloaded 350,000 rows
2016: downloaded 400,000 rows
2016: downloaded 404,062 rows

2016 complete
Shape: (404062, 13)
Date range: 2016-01-01T00:01:02.000 to 2016-12-31T23:58:51.000


In [ ]:
API_URL_CFS_2015 = "https://data.nola.gov/resource/w68y-xmk6.json"

df_cfs_2015 = download_cfs_year(
    API_URL_CFS_2015,
    2015,
    KEEP_COLS_CFS
)
df_cfs_2015.to_parquet(
    "CFS_2015_raw_selected.parquet",
    index=False
)

2015: downloaded 50,000 rows
2015: downloaded 100,000 rows
2015: downloaded 150,000 rows
2015: downloaded 200,000 rows
2015: downloaded 250,000 rows
2015: downloaded 300,000 rows
2015: downloaded 350,000 rows
2015: downloaded 400,000 rows
2015: downloaded 432,733 rows

2015 complete
Shape: (432733, 13)
Date range: 2015-01-01T00:00:34.000 to 2015-12-31T11:59:30.000


In [ ]:
API_URL_CFS_2014 = "https://data.nola.gov/resource/jsyu-nz5r.json"

df_cfs_2014 = download_cfs_year(
    API_URL_CFS_2014,
    2014,
    KEEP_COLS_CFS
)
df_cfs_2014.to_parquet(
    "CFS_2014_raw_selected.parquet",
    index=False
)

2014: downloaded 50,000 rows
2014: downloaded 100,000 rows
2014: downloaded 150,000 rows
2014: downloaded 200,000 rows
2014: downloaded 250,000 rows
2014: downloaded 300,000 rows
2014: downloaded 350,000 rows
2014: downloaded 400,000 rows
2014: downloaded 447,462 rows

2014 complete
Shape: (447462, 13)
Date range: 2014-01-01T00:00:14.000 to 2014-12-31T11:58:29.000


In [ ]:
API_URL_CFS_2013 = "https://data.nola.gov/resource/5fn8-vtui.json"

df_cfs_2013 = download_cfs_year(
    API_URL_CFS_2013,
    2013,
    KEEP_COLS_CFS
)
df_cfs_2013.to_parquet(
    "CFS_2013_raw_selected.parquet",
    index=False
)

HTTPError: 400 Client Error: Bad Request for url: https://data.nola.gov/resource/5fn8-vtui.json?%24select=nopd_item%2Ctype_%2Ctypetext%2Cinitialtype%2Cinitialtypetext%2Cpriority%2Cinitialpriority%2Ctimecreate%2Cselfinitiated%2Cblock_address%2Czip%2Cpolicedistrict%2Clocation&%24limit=50000&%24offset=0&%24order=timecreate%2Cnopd_item

In [ ]:
API_URL_CFS_2013 = "https://data.nola.gov/resource/5fn8-vtui.json"
r = requests.get(
    API_URL_CFS_2013,
    params={"$limit": 10},
    timeout=120
)

print(r.status_code)
print(r.text[:2000])

200
[{"nopd_item":"A0000113","type_":"94","typetext":"DISCHARGING FIREARM","priority":"2B","mapx":"3696313.00000000","mapy":"533332.00000000","timecreate":"2012-12-31T23:59:34.000","timedispatch":"2013-01-01T00:03:10.000","timearrive":"2013-01-01T00:23:57.000","timeclosed":"2013-01-01T00:24:10.000","disposition":"UNF","dispositiontext":"UNFOUNDED","block_address":"052XX Burgundy St","zip":"70117","policedistrict":"5","location":{"latitude":"29.960019973023","longitude":"-90.021230929534"},":@computed_region_evki_aju8":"8",":@computed_region_u4yh_3wk9":"7075",":@computed_region_7fw3_kdpf":"32",":@computed_region_spev_d8jm":"3766",":@computed_region_m56f_hbma":"153",":@computed_region_sikx_bdeb":"153",":@computed_region_ewbu_t8bu":"7075",":@computed_region_k37d_then":"5"}
,{"nopd_item":"A0000213","type_":"94","typetext":"DISCHARGING FIREARM","priority":"2B","mapx":"3710263.00000000","mapy":"518976.00000000","timecreate":"2012-12-31T23:59:49.000","timedispatch":"2013-01-01T00:05:43.000","

In [ ]:
params = {
    "$select": ",".join(KEEP_COLS_CFS),
    "$limit": 10
}

r = requests.get(API_URL_CFS_2013, params=params, timeout=120)

print(r.status_code)
print(r.text[:2000])

400
{"message":"Query coordinator error: query.soql.no-such-column; No such column: initialtype; position: Map(row -> 1, column -> 42, line -> \"SELECT `nopd_item`, `type_`, `typetext`, `initialtype`, `initialtypetext`, `priority`, `initialpriority`, `timecreate`, `selfinitiated`, `block_address`, `zip`, `policedistrict`, `location` LIMIT 10\\n                                         ^\")","errorCode":"query.soql.no-such-column","data":{"column":"initialtype","dataset":"foxtrot.25115","position":{"row":1,"column":42,"line":"SELECT `nopd_item`, `type_`, `typetext`, `initialtype`, `initialtypetext`, `priority`, `initialpriority`, `timecreate`, `selfinitiated`, `block_address`, `zip`, `policedistrict`, `location` LIMIT 10\n                                         ^"}}}


In [ ]:
KEEP_COLS_CFS_2013 = [
    "nopd_item",
    "type_",
    "typetext",
    "priority",
    "timecreate",
    "block_address",
    "zip",
    "policedistrict",
    "location",
]

API_URL_CFS_2013 = "https://data.nola.gov/resource/5fn8-vtui.json"

df_cfs_2013 = download_cfs_year(
    API_URL_CFS_2013,
    2013,
    KEEP_COLS_CFS_2013
)
df_cfs_2013.to_parquet(
    "CFS_2013_raw_selected.parquet",
    index=False
)

2013: downloaded 50,000 rows
2013: downloaded 100,000 rows
2013: downloaded 150,000 rows
2013: downloaded 200,000 rows
2013: downloaded 250,000 rows
2013: downloaded 300,000 rows
2013: downloaded 350,000 rows
2013: downloaded 400,000 rows
2013: downloaded 450,000 rows
2013: downloaded 463,619 rows

2013 complete
Shape: (463619, 9)
Date range: 2012-12-31T23:59:34.000 to 2013-12-31T23:59:38.000


In [ ]:
KEEP_COLS_CFS_2012 = [
    "nopd_item",
    "type",
    "typetext",
    "priority",
    "timecreate",
    "block_address",
    "zip",
    "policedistrict",
    "location",
]

API_URL_CFS_2012 = "https://data.nola.gov/resource/rv3g-ypg7.json"

df_cfs_2012 = download_cfs_year(
    API_URL_CFS_2012,
    2012,
    KEEP_COLS_CFS_2012
)
df_cfs_2012.to_parquet(
    "CFS_2012_raw_selected.parquet",
    index=False
)

2012: downloaded 50,000 rows
2012: downloaded 100,000 rows
2012: downloaded 150,000 rows
2012: downloaded 200,000 rows
2012: downloaded 250,000 rows
2012: downloaded 300,000 rows
2012: downloaded 350,000 rows
2012: downloaded 400,000 rows
2012: downloaded 450,000 rows
2012: downloaded 500,000 rows
2012: downloaded 505,012 rows

2012 complete
Shape: (505012, 9)
Date range: 2012-01-01T00:00:11.000 to 2012-12-31T23:59:39.000


In [ ]:
from google.colab import files

files.download("CFS_2023_raw_selected.parquet")

# NOLA Electronic Police Reports

In [ ]:
import requests
import pandas as pd

In [ ]:
KEEP_COLS_EPR = [
    "item_number",
    "district",
    "location",
    "disposition",
    "signal_type",
    "signal_description",
    "occurred_date_time",
    "report_type",
    "offender_race",
    "offender_gender",
    "offender_age",
]

In [ ]:
def download_epr_year(api_url, year, keep_cols, limit=50000):

    offset = 0
    parts = []

    while True:

        params = {
            "$select": ",".join(keep_cols),
            "$limit": limit,
            "$offset": offset,
            "$order": "occurred_date_time,item_number"
        }

        r = requests.get(api_url, params=params, timeout=120)
        r.raise_for_status()

        rows = r.json()

        if not rows:
            break

        batch = pd.DataFrame(rows)
        parts.append(batch)

        offset += len(batch)

        print(f"{year}: downloaded {offset:,} rows")

        if len(batch) < limit:
            break

    df = pd.concat(parts, ignore_index=True)

    print(f"\n{year} complete")
    print("Shape:", df.shape)
    print(
        "Date range:",
        df["occurred_date_time"].min(),
        "to",
        df["occurred_date_time"].max()
    )

    return df

In [ ]:
def basic_epr_checks(df):

    print("Shape:", df.shape)

    print("\nDate range:")
    print(
        df["occurred_date_time"].min(),
        "to",
        df["occurred_date_time"].max()
    )

    print("\nDuplicate Item_Number:")
    print(df["item_number"].duplicated().sum())

    print("\nMissingness:")
    print(df.isna().sum().sort_values(ascending=False))

    print("\nTop signal descriptions:")
    print(
        df["signal_description"]
        .value_counts(dropna=False)
        .head(20)
    )

    print("\nReport types:")
    print(
        df["report_type"]
        .value_counts(dropna=False)
    )

    print("\nDistricts:")
    print(
        df["district"]
        .value_counts(dropna=False)
        .sort_index()
    )

In [ ]:
API_URL_EPR_2023 = "https://data.nola.gov/resource/j3gz-62a2.json"

YEAR = 2023

df_epr = download_epr_year(
    API_URL_EPR_2023,
    YEAR,
    KEEP_COLS_EPR
)

basic_epr_checks(df_epr)

df_epr.to_parquet(
    f"EPR_{YEAR}_raw_selected.parquet",
    index=False
)

2023: downloaded 50,000 rows
2023: downloaded 100,000 rows
2023: downloaded 150,000 rows
2023: downloaded 200,000 rows
2023: downloaded 250,000 rows
2023: downloaded 300,000 rows
2023: downloaded 312,645 rows

2023 complete
Shape: (312645, 11)
Date range: 2023-01-01 00:00:00.000 to 2023-12-31 23:57:00.000
Shape: (312645, 11)

Date range:
2023-01-01 00:00:00.000 to 2023-12-31 23:57:00.000

Duplicate Item_Number:
253457

Missingness:
offender_age          137980
offender_race          71872
offender_gender        71872
location                   0
district                   0
item_number                0
disposition                0
occurred_date_time         0
signal_description         0
signal_type                0
report_type                0
dtype: int64

Top signal descriptions:
signal_description
DISTURBANCE (DOMESTIC)           33503
AUTO THEFT                       22883
MISCELLANEOUS INCIDENT           21546
SIMPLE BATTERY (DOMESTIC)        21485
SIMPLE BURGLARY (VEHICLE)      

In [ ]:
dup = df_epr[
    df_epr["item_number"].duplicated(keep=False)
].copy()

print("Total rows:", len(df_epr))
print("Unique item numbers:", df_epr["item_number"].nunique())
print("Rows with duplicated item numbers:", len(dup))
print("Duplicated item numbers:", dup["item_number"].nunique())

Total rows: 312645
Unique item numbers: 59188
Rows with duplicated item numbers: 294908
Duplicated item numbers: 41451


In [ ]:
CORE = [
    "occurred_date_time",
    "location",
    "district",
    "signal_type",
    "signal_description",
]
summary = (
    dup
    .groupby("item_number")[CORE]
    .nunique(dropna=False)
)

summary.head()

,occurred_date_time,location,district,signal_type,signal_description
item_number,,,,,
A-00032-23,1,1,1,1,1
A-00048-23,1,1,1,1,1
A-00055-24,2,1,1,1,1
A-00060-23,1,1,1,1,1
A-00062-24,2,1,1,1,1


In [ ]:
print(
    "\nDuplicate item numbers with disagreement:"
)

print(
    (summary > 1).sum()
)


Duplicate item numbers with disagreement:
occurred_date_time    29396
location                 41
district                  7
signal_type              47
signal_description       47
dtype: int64


In [ ]:
dup.groupby("item_number")["report_type"].nunique().value_counts()

,count
report_type,
1,40513
2,938


So supplements explain some duplication—but clearly not most: most duplicates got to be driven by multiple offenders, victims, or charges.
The same-item differences in locations/districts are tiny.
But why do 29,396 item numbers have multiple occurrence times?


In [ ]:
changed_time = summary.index[
    summary["occurred_date_time"] > 1
]

dup[
    dup["item_number"].isin(changed_time[:10])
].sort_values(
    ["item_number", "occurred_date_time"]
)

,item_number,district,location,disposition,signal_type,signal_description,occurred_date_time,report_type,offender_race,offender_gender,offender_age
312506,A-00055-24,5,N Tonti St & Franklin Av,OPEN,62C,SIMPLE BURGLARY (VEHICLE),2023-12-31 22:45:00,Incident Report,UNKNOWN,UNKNOWN,NaN
312507,A-00055-24,5,N Tonti St & Franklin Av,OPEN,62C,SIMPLE BURGLARY (VEHICLE),2023-12-31 22:45:00,Incident Report,UNKNOWN,UNKNOWN,NaN
312508,A-00055-24,5,N Tonti St & Franklin Av,OPEN,62C,SIMPLE BURGLARY (VEHICLE),2023-12-31 22:45:00,Incident Report,UNKNOWN,UNKNOWN,NaN
312509,A-00055-24,5,N Tonti St & Franklin Av,OPEN,62C,SIMPLE BURGLARY (VEHICLE),2023-12-31 22:45:00,Incident Report,UNKNOWN,UNKNOWN,NaN
312510,A-00055-24,5,N Tonti St & Franklin Av,OPEN,62C,SIMPLE BURGLARY (VEHICLE),2023-12-31 22:45:00,Incident Report,UNKNOWN,UNKNOWN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
312211,A-00163-24,7,129XX Chef Menteur Hw,OPEN,67S,SHOPLIFTING,2023-12-31 21:22:00,Incident Report,BLACK,FEMALE,NaN
312212,A-00163-24,7,129XX Chef Menteur Hw,OPEN,67S,SHOPLIFTING,2023-12-31 21:22:00,Incident Report,BLACK,FEMALE,NaN
312213,A-00163-24,7,129XX Chef Menteur Hw,OPEN,67S,SHOPLIFTING,2023-12-31 21:22:00,Incident Report,BLACK,FEMALE,NaN
312214,A-00163-24,7,129XX Chef Menteur Hw,OPEN,67S,SHOPLIFTING,2023-12-31 21:22:00,Incident Report,BLACK,FEMALE,NaN


In [ ]:
df_epr["occurred_date_time"] = pd.to_datetime(
    df_epr["occurred_date_time"]
)

ValueError: time data "2023-01-01 00:10:00" doesn't match format "%Y-%m-%d %H:%M:%S.%f", at position 3. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

okay, so at least many of those differences are due to time formatting issues: some are 2023-01-01 00:10:00, for example, while some are 2023-12-31 21:22:00.000

In [ ]:
df_epr["occurred_date_time"] = pd.to_datetime(
    df_epr["occurred_date_time"],
    format="mixed"
)

In [ ]:
#recreate dup copy
dup = df_epr[
    df_epr["item_number"].duplicated(keep=False)
].copy()

summary = (
    dup
    .groupby("item_number")[CORE]
    .nunique(dropna=False)
)

#check disagreements again
(summary > 1).sum()

,0
occurred_date_time,52
location,41
district,7
signal_type,47
signal_description,47


therefoe,a mong 41,451 duplicated item_numbers, now

* 29396—>only 52 have different occurrence times (0.13%)
* 41 have different locations (0.10%)
* 7 have different districts (0.02%)
* 47 have different signal types/descriptions (0.11%)

need to specify the XX in street block address and create deterministic geocoder input

# NOLA EPR Reusable Download and Cleaning

In [2]:
import requests
import pandas as pd
import re

In [23]:
GEOCODER_URL = (
    "https://geocoding.geo.census.gov/"
    "geocoder/geographies/addressbatch"
)

GEOCODER_COLUMNS = [
    "geocode_id",
    "input_address",
    "match_status",
    "match_type",
    "matched_address",
    "coordinates",
    "tigerline_id",
    "side",
    "state_fips",
    "county_fips",
    "tract",
    "block"
]

In [3]:
KEEP_COLS_EPR = [
    "item_number",
    "district",
    "location",
    "signal_type",
    "signal_description",
    "occurred_date_time",
    "report_type",
    "hate_crime",
    "offender_race",
    "offender_gender",
    "offender_age",
]


def download_epr_year(api_url, year, keep_cols=KEEP_COLS_EPR, limit=50000):
    offset = 0
    parts = []

    while True:
        params = {
            "$select": ",".join(keep_cols),
            "$limit": limit,
            "$offset": offset,
            "$order": "occurred_date_time,item_number"
        }

        r = requests.get(api_url, params=params, timeout=120)
        r.raise_for_status()

        rows = r.json()

        if not rows:
            break

        batch = pd.DataFrame(rows)
        parts.append(batch)

        offset += len(batch)
        print(f"{year}: downloaded {offset:,} rows")

        if len(batch) < limit:
            break

    df = pd.concat(parts, ignore_index=True)

    print(f"\n{year} complete")
    print("Shape:", df.shape)
    print(
        "Date range:",
        df["occurred_date_time"].min(),
        "to",
        df["occurred_date_time"].max()
    )

    return df

In [4]:
def fill_block_address(address):
    if pd.isna(address):
        return address

    match = re.match(r"^([0-9Xx]+)(\s+.*)$", address)

    if not match:
        return address

    number = (
        match.group(1)
        .replace("X", "0")
        .replace("x", "0")
    )

    return number + match.group(2)


def prepare_epr_for_geocoding(df):
    out = df.copy()

    #just conservative timestamp normalization
    out["occurred_date_time"] = pd.to_datetime(
        out["occurred_date_time"],
        format="mixed",
        errors="raise"
    )

    #conservative address normalization
    out["location_clean"] = (
        out["location"]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    #masked-block completion: XX -> 00
    out["filled_block_address"] = (
        out["location_clean"]
        .apply(fill_block_address)
    )

    out["city"] = "New Orleans" #geocoding merge needs city and state
    out["state"] = "LA"

    return out

Note that to make it work for batch geocoding, the table may need to be designed to have City and State in separate columns. Seems that the Census batch format is not a single concatenated address string—the expected CSV format is 'Unique ID, Street address, City, State, ZIP' where City, State, and ZIP each occupy their own field. ZIP can be blank.

In [5]:
def quick_epr_check(df):
    print("Shape:", df.shape)

    print("\nDate range:")
    print(
        df["occurred_date_time"].min(),
        "to",
        df["occurred_date_time"].max()
    )

    print("\nDuplicate item_number rows:")
    print(df["item_number"].duplicated().sum())

    print("\nMissingness:")
    print(df.isna().sum().sort_values(ascending=False))

    print("\nReport types:")
    print(df["report_type"].value_counts(dropna=False))

    print("\nTop signal descriptions:")
    print(
        df["signal_description"]
        .value_counts(dropna=False)
        .head(20)
    )

In [6]:
# then just make the year-specific calls
YEAR = 2025
API_URL_EPR = "https://data.nola.gov/resource/agqi-9adb.json"

df_epr = download_epr_year(
    API_URL_EPR,
    YEAR
)

quick_epr_check(df_epr)

2025: downloaded 50,000 rows
2025: downloaded 51,195 rows

2025 complete
Shape: (51195, 11)
Date range: 2025-01-01 00:15:00 to 2025-11-06 06:43:00
Shape: (51195, 11)

Date range:
2025-01-01 00:15:00 to 2025-11-06 06:43:00

Duplicate item_number rows:
21032

Missingness:
hate_crime            51181
offender_age          36685
offender_race         22252
offender_gender       22250
location                  0
item_number               0
district                  0
report_type               0
occurred_date_time        0
signal_description        0
signal_type               0
dtype: int64

Report types:
report_type
Incident Report        41627
Supplemental Report     9568
Name: count, dtype: int64

Top signal descriptions:
signal_description
DISTURBANCE (DOMESTIC)        7518
MISCELLANEOUS INCIDENT        4737
SHOPLIFTING                   3596
DISTURBANCE                   3085
SIMPLE BATTERY (DOMESTIC)     2751
THEFT                         2329
SIMPLE BATTERY                2050
SIMPLE 

In [ ]:
# untouched selected raw extract
df_epr.to_parquet(
    f"EPR_{YEAR}_raw_selected.parquet",
    index=False
)

In [7]:
# the minimally cleaned geocoding copy
df_epr_geocode = prepare_epr_for_geocoding(df_epr)

In [8]:
print("Datetime dtype:")
print(df_epr_geocode["occurred_date_time"].dtype)

print("\nAddresses changed by masked-digit filling:")
print(
    (
        df_epr_geocode["location_clean"]
        != df_epr_geocode["filled_block_address"]
    ).sum()
)

df_epr_geocode[
    ["location", "location_clean", "filled_block_address"]
].drop_duplicates().head(20)

Datetime dtype:
datetime64[ns]

Addresses changed by masked-digit filling:
47283


,location,location_clean,filled_block_address
0,Decatur & Toulouse,Decatur & Toulouse,Decatur & Toulouse
1,007XX Decatur St,007XX Decatur St,00700 Decatur St
3,002XX Cypress Grove Ct,002XX Cypress Grove Ct,00200 Cypress Grove Ct
4,003XX Bourbon St,003XX Bourbon St,00300 Bourbon St
5,072XX Culpepper Dr,072XX Culpepper Dr,07200 Culpepper Dr
6,006XX Lesseps St,006XX Lesseps St,00600 Lesseps St
7,012XX Canal St,012XX Canal St,01200 Canal St
8,019XX Tchoupitoulas St,019XX Tchoupitoulas St,01900 Tchoupitoulas St
9,016XX Oretha Bd,016XX Oretha Bd,01600 Oretha Bd
13,027XX Napoleon Av,027XX Napoleon Av,02700 Napoleon Av


In [9]:
print(
    (df_epr_geocode["location"] !=
     df_epr_geocode["filled_block_address"]).sum()
)

47283


In [11]:
address_lookup = (
    df_epr_geocode[
        ["filled_block_address", "city", "state"]
    ]
    .drop_duplicates()
)

In [12]:
df_epr_geocode.to_parquet(
    f"EPR_{YEAR}_geocode_ready.parquet",
    index=False
)

address_lookup.to_parquet(
    f"EPR_{YEAR}_address_lookup.parquet",
    index=False
)

In [14]:
'''from google.colab import files

files.download(
    f"EPR_{YEAR}_address_lookup.parquet"
)
from google.colab import files

files.download(
    f"EPR_{YEAR}_geocode_ready.parquet"
)'''

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
address_lookup_matched = geocode_address_lookup( #now that I have the address look up table and the geocode ready df to merge back into
    address_lookup
)

In [ ]:
df_epr_tract = df_epr_geocode.merge(
    address_lookup_matched[
        [
            "filled_block_address",
            "city",
            "state",
            "match_status",
            "match_type",
            "matched_address",
            "coordinates",
            "state_fips",
            "county_fips",
            "tract",
            "block"
        ]
    ],
    on=["filled_block_address", "city", "state"],
    how="left",
    validate="many_to_one"
)

In [ ]:
print("Rows before merge:", len(df_epr_geocode))
print("Rows after merge: ", len(df_epr_tract))

print("\nMatch status:")
print(df_epr_tract["match_status"].value_counts(dropna=False))

print("\nMissing tract:")
print(df_epr_tract["tract"].isna().sum())

In [ ]:
df_epr_tract.to_parquet(
    f"EPR_{YEAR}_tract_coded.parquet",
    index=False
)

In [ ]:
from google.colab import files

files.download(
    f"EPR_{YEAR}_tract_coded.parquet"
)

# Geo-Coding Pilot Records

In [15]:
import pandas as pd
import requests
from io import StringIO

In [16]:
pilot = (
    address_lookup
    .head(5)
    .reset_index(drop=True)
    .copy()
)

pilot["id"] = pilot.index + 1
pilot["zip"] = ""

In [17]:
pilot_batch = pilot[
    ["id", "filled_block_address", "city", "state", "zip"]
]

pilot_batch

,id,filled_block_address,city,state,zip
0,1,Decatur & Toulouse,New Orleans,LA,
1,2,00700 Decatur St,New Orleans,LA,
2,3,00200 Cypress Grove Ct,New Orleans,LA,
3,4,00300 Bourbon St,New Orleans,LA,
4,5,07200 Culpepper Dr,New Orleans,LA,


In [18]:
#write a csv, in case there's format preference?
pilot_batch.to_csv(
    "epr_geocoder_pilot.csv",
    index=False,
    header=False
)

In [19]:
GEOCODER_URL = (
    "https://geocoding.geo.census.gov/"
    "geocoder/geographies/addressbatch"
)

with open("epr_geocoder_pilot.csv", "rb") as f:
    response = requests.post(
        GEOCODER_URL,
        files={"addressFile": f},
        data={
            "benchmark": "Public_AR_Current",
            "vintage": "Current_Current"
        },
        timeout=120
    )

response.raise_for_status()

print(response.text)

"1","Decatur & Toulouse, New Orleans, LA, ","Match","Non_Exact","DECATUR ST & TOULOUSE ST, NEW ORLEANS, LA, 70130","-90.063466889596,29.955900585277","","","22","071","013502","2011"
"2","00700 Decatur St, New Orleans, LA, ","Match","Exact","700 DECATUR ST, NEW ORLEANS, LA, 70116","-90.062649862424,29.956731376033","191753378","R","22","071","013502","2003"
"3","00200 Cypress Grove Ct, New Orleans, LA, ","Match","Exact","200 CYPRESS GROVE CT, NEW ORLEANS, LA, 70131","-89.994421892677,29.90035546367","639237871","R","22","071","000617","3002"
"4","00300 Bourbon St, New Orleans, LA, ","Match","Exact","300 BOURBON ST, NEW ORLEANS, LA, 70130","-90.068328410487,29.955504311786","191710840","R","22","071","013502","1018"
"5","07200 Culpepper Dr, New Orleans, LA, ","Match","Exact","7200 CULPEPPER DR, NEW ORLEANS, LA, 70126","-90.017686024499,30.028739471873","191704990","L","22","071","001724","3008"



In [20]:
result = pd.read_csv(
    StringIO(response.text),
    header=None
)

result

,0,1,2,3,4,5,6,7,8,9,10,11
0,1,"Decatur & Toulouse, New Orleans, LA,",Match,Non_Exact,"DECATUR ST & TOULOUSE ST, NEW ORLEANS, LA, 70130","-90.063466889596,29.955900585277",NaN,NaN,22,71,13502,2011
1,2,"00700 Decatur St, New Orleans, LA,",Match,Exact,"700 DECATUR ST, NEW ORLEANS, LA, 70116","-90.062649862424,29.956731376033",191753378.0,R,22,71,13502,2003
2,3,"00200 Cypress Grove Ct, New Orleans, LA,",Match,Exact,"200 CYPRESS GROVE CT, NEW ORLEANS, LA, 70131","-89.994421892677,29.90035546367",639237871.0,R,22,71,617,3002
3,4,"00300 Bourbon St, New Orleans, LA,",Match,Exact,"300 BOURBON ST, NEW ORLEANS, LA, 70130","-90.068328410487,29.955504311786",191710840.0,R,22,71,13502,1018
4,5,"07200 Culpepper Dr, New Orleans, LA,",Match,Exact,"7200 CULPEPPER DR, NEW ORLEANS, LA, 70126","-90.017686024499,30.028739471873",191704990.0,L,22,71,1724,3008


In [22]:
result.columns = [
    "id",
    "input_address",
    "match_status",
    "match_type",
    "matched_address",
    "coordinates",
    "tigerline_id",
    "side",
    "state_fips",
    "county_fips",
    "tract",
    "block"
]

result

,id,input_address,match_status,match_type,matched_address,coordinates,tigerline_id,side,state_fips,county_fips,tract,block
0,1,"Decatur & Toulouse, New Orleans, LA,",Match,Non_Exact,"DECATUR ST & TOULOUSE ST, NEW ORLEANS, LA, 70130","-90.063466889596,29.955900585277",NaN,NaN,22,71,13502,2011
1,2,"00700 Decatur St, New Orleans, LA,",Match,Exact,"700 DECATUR ST, NEW ORLEANS, LA, 70116","-90.062649862424,29.956731376033",191753378.0,R,22,71,13502,2003
2,3,"00200 Cypress Grove Ct, New Orleans, LA,",Match,Exact,"200 CYPRESS GROVE CT, NEW ORLEANS, LA, 70131","-89.994421892677,29.90035546367",639237871.0,R,22,71,617,3002
3,4,"00300 Bourbon St, New Orleans, LA,",Match,Exact,"300 BOURBON ST, NEW ORLEANS, LA, 70130","-90.068328410487,29.955504311786",191710840.0,R,22,71,13502,1018
4,5,"07200 Culpepper Dr, New Orleans, LA,",Match,Exact,"7200 CULPEPPER DR, NEW ORLEANS, LA, 70126","-90.017686024499,30.028739471873",191704990.0,L,22,71,1724,3008
